# Track 10 — Capstone: Meeting Minutes → Actions (회의록 → 액션아이템)

## 회의록 → 구조화 `action_items` 추출이란?

회의록에는 후속 작업이 자연어 문장 속에 흩어져 있습니다. 이 캡스톤은 회의록에서 `meeting_title`과 `action_items[*].{assignee, description, due, priority}`를 뽑아 **검증 가능한 JSON**으로 만듭니다.

신뢰할 수 있는 추출은 세 단계로 만듭니다.

- **프롬프트:** 필요한 키와 형식을 명확히 지시합니다.
- **검증:** `StructuredOutputPipeline`으로 JSON 추출과 스키마 검사를 수행합니다.
- **복구:** 검증 오류를 모델에 다시 전달해 스키마에 맞게 재출력합니다.

## 이 노트북에서 보여줄 것

| Session | 보여주는 것 | 목적 |
|---|---|---|
| 1. Setup | facade 시작 · 골든 로더 · 정적 메트릭 데모 · 패키지 저장 헬퍼 | 공통 준비 |
| 2. 왜 파이프라인인가 | `json.loads`와 `StructuredOutputPipeline` 비교 | 원문 파싱만으로 부족한 이유 확인 |
| 3. STRICT → LOOSE → REPAIR | 프롬프트 강도별 추출과 복구 | 추출·검증·복구 운영 루프 확인 |
| 4. 패키지 | 정적 회귀와 라이브 루프 결과 저장 | 제출·회귀 산출물 마감 |

## 이 노트북을 마치면

- 회의록을 액션아이템 JSON으로 추출하고 스키마 위반을 탐지할 수 있습니다.
- 느슨한 출력이 깨졌을 때 검증 오류를 활용해 재프롬프트로 복구할 수 있습니다.
- 정적 메트릭 데모와 라이브 추출 결과를 구분해 패키지로 정리할 수 있습니다.

**산출물:** `_out/02/capstone_package.json`  
**실행 조건:** Session 1·2·4는 API 키 없이 실행됩니다. Session 3 라이브 루프는 EXAONE API 키가 필요합니다.

> 요약: 회의록 추출을 프롬프트·검증·복구 루프로 안정화하고, 결과를 회귀 패키지로 마감하는 캡스톤입니다.


## Session 1. Setup


### Session 1-1. Setup

**하는 일:** 경로·클라이언트·골든 로더·정적 메트릭 데모·패키지 저장 헬퍼를 준비합니다.

**정상:** `exaone 0.1.0 | model LGAI-EXAONE/K-EXAONE-236B-A23B | HAS_API True/False` 한 줄이 출력됩니다.

**의미:** 이후 단계에서 쓸 경로·클라이언트가 맞는지 먼저 봅니다.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import logging

# (en) Quiet library logs so the notebook output stays readable.
# (kr) 라이브러리 로그를 줄여 노트북 출력을 읽기 쉽게 한다.
for _log_name in ("exaone", "exaone.llm", "exaone.llm.exaone_client", "urllib3"):
    logging.getLogger(_log_name).setLevel(logging.ERROR)

# (en) Facade-only startup; requires editable install at the repo root.
# (kr) `exaone` facade로 시작한다. 저장소 루트에서 editable 설치가 필요하다.
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "`exaone`이 설치되지 않았습니다. 저장소 루트에서 "
        "pip install -r requirements.txt && pip install -e ./exaone 후 커널을 재시작하세요."
    ) from exc

exaone.load_project_env()
ROOT = exaone.project_root()
TRACK10 = ROOT / "recipes" / "track10_ax_capstones"
DATA = TRACK10 / "data"
API_KEY = os.environ.get("EXAONE_API_KEY", "").strip()
BASE_URL = os.environ.get("EXAONE_BASE_URL", "").strip() or "http://localhost:8000/v1"
MODEL = os.environ.get("EXAONE_MODEL", "").strip() or exaone.llm.ExaoneClient.DEFAULT_MODEL
HAS_API = bool(API_KEY)
client = None
if HAS_API:
    client = exaone.llm.ExaoneAPIClient(base_url=BASE_URL, model=MODEL, api_key=API_KEY)
print("exaone", exaone.__version__, "| model", MODEL, "| HAS_API", HAS_API)

def load_capstone_golden(tag: str) -> list[dict]:
    # (en) Load golden rows for this capstone id or shared "all" rows.
    # (kr) 해당 캡스톤과 공통 "all" 골든 사례를 함께 불러온다.
    rows: list[dict] = []
    for line in (DATA / "capstone_golden.jsonl").read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        if row.get("capstone") in (tag, "all"):
            rows.append(row)
    return rows


def regression_m1_m6_m9(rows: list[dict]) -> dict:
    # (en) Static fixture metric demo (M1/M6/M9) over golden rows — metric MECHANICS, not live agent.
    # (kr) 정적 골든 fixture로 M1/M6/M9를 계산한다. 라이브 에이전트 성능이 아니라 메트릭 동작 예시다.
    from eval.metrics.m1_task_success import TaskGold
    from eval.metrics.m6_schema_adherence import SchemaSpec
    from eval.metrics.m9_faithfulness import LengthRatioJudge
    from eval.metrics import m1_task_success, m6_schema_adherence
    from eval.metrics.types import TrialResult

    m1s, m6s, m9s = [], [], []
    cases = []
    for row in rows:
        tid = row["id"]
        content = row.get("trial_content") or str(row.get("expected_answer", ""))
        tr = TrialResult(
            trial_id=f"cap-{tid}",
            task_id=tid,
            dataset="track10.golden",
            runner="capstone",
            final_content=content,
        )
        m1 = m6 = m9 = None
        if row.get("expected_answer") is not None:
            m1 = m1_task_success.score_trial_exact(tr, TaskGold(task_id=tid, answer=row["expected_answer"]))
            m1s.append(m1)
        rk = row.get("required_keys")
        if rk:
            _, loose = m6_schema_adherence.score_trial(tr, SchemaSpec(required_keys=rk))
            m6 = loose
            m6s.append(1.0 if loose else 0.0)
        if row.get("grounding_context"):
            m9 = LengthRatioJudge()(trial=tr, gold={"context": row["grounding_context"]})
            m9s.append(m9)
        cases.append({"id": tid, "M1": m1, "M6": m6, "M9": m9})
    mean = lambda xs: sum(xs) / len(xs) if xs else 0.0
    return {"n": len(rows), "M1_mean": mean(m1s), "M6_loose_mean": mean(m6s), "M9_mean": mean(m9s), "cases": cases}


def save_package(capstone_nb: str, body: dict) -> Path:
    # (en) Write capstone_package.json under <track>/_out/<nb>/ (absolute path, CWD-independent).
    # (kr) 절대경로로 <track>/_out/<nb>/capstone_package.json을 저장한다(커널 CWD와 무관).
    out_dir = TRACK10 / "_out" / capstone_nb
    out_dir.mkdir(parents=True, exist_ok=True)
    slo = exaone.observability.SLOSpec(
        name=f"capstone-{capstone_nb}",
        p95_chat_latency_ms=8000,
        structured_output_success_min="95%",
        notes="Track 10 capstone — adjust per deployment.",
    )
    payload = {
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "capstone_id": capstone_nb,
        "slo": slo.to_dict(),
        **body,
    }
    path = out_dir / "capstone_package.json"
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print("saved", path.resolve())
    return path


**출력 해석:** `exaone 0.1.0 | model LGAI-EXAONE/K-EXAONE-236B-A23B | HAS_API True/False` 한 줄이 보이면 client·DATA가 준비된 것입니다(키가 있으면 `True`, 이번 tier 에서는 보통 `True`).

- `HAS_API=True`이면 Session 3 라이브 1건이 실행되고, `False`이면 깔끔히 건너뜁니다(Session 1·2·4 정적 메트릭 데모는 키 없이 동작).
- `client`와 `TRACK10`·`DATA`가 **절대경로**로 잡혀, 이후 셀이 어느 CWD에서 실행돼도 같은 fixture를 읽고 같은 위치에 패키지를 저장합니다.

## Session 2. 왜 `json.loads`가 아니라 `StructuredOutputPipeline` 인가 (오프라인)

**무엇을 확인하나** — 모델 출력을 그냥 `json.loads` 하면 **두 가지로 틀립니다**: 펜스·산문이 섞이면 **깨지고**(JSONDecodeError), 키가 틀려도 **그냥 통과**시킵니다. `StructuredOutputPipeline`은 펜스·산문에서 JSON을 **추출**하고 스키마로 **검증**해 둘 다 바로잡습니다. 같은 원문 3종(깔끔/펜스/키 변형)으로 `json.loads` vs 파이프라인을 대비합니다. 이 입력들은 느슨한 프롬프트일 때 EXAONE이 실제로 내는 출력 모양(Session 3)을 본뜬 것이고, 키 없이 실행됩니다.

### Session 2-1. `json.loads`는 두 가지로 틀리고, 파이프라인은 둘 다 잡는다

**하는 일:** 원문 문자열 3종(깔끔한 JSON · ` ```json ` 펜스 · 키 변형)을 naive `json.loads`와 `StructuredOutputPipeline` 양쪽에 넣어 비교합니다.

**정상:** `회의록 fixture 2 건: ['m01', 'm02']` 다음에
- `json.loads vs 파이프라인: [{'case': '깔끔한 JSON', 'json_loads': 'ok', 'pipeline_ok': True, 'error': ''}, {'case': '```json 펜스', 'json_loads': '✗ JSONDecodeError', 'pipeline_ok': True, 'error': ''}, {'case': '키 변형(meeting/task)', 'json_loads': 'ok', 'pipeline_ok': False, 'error': "'meeting_title' is a required property"}]`

**의미:** 펜스엔 `json.loads`가 깨지지만 파이프라인은 추출해 통과(`pipeline_ok=True`), 키 변형은 `json.loads`가 그냥 통과시키지만 파이프라인은 거부(`pipeline_ok=False`) — 파이프라인의 효용이 **양방향**으로 드러납니다.

In [ ]:
minutes_data = json.loads((DATA / "meeting_minutes.json").read_text(encoding="utf-8"))
_samples = minutes_data["samples"]
print("회의록 fixture", len(_samples), "건:", [s["id"] for s in _samples])
ACTION_SCHEMA = {
    "type": "object",
    "required": ["meeting_title", "action_items"],
    "properties": {
        "meeting_title": {"type": "string"},
        "action_items": {
            "type": "array",
            "items": {
                "type": "object",
                "required": ["assignee", "description", "due", "priority"],
                "properties": {
                    "assignee": {"type": "string"},
                    "description": {"type": "string"},
                    "due": {"type": "string"},
                    "priority": {"type": "string", "enum": ["high", "medium", "low"]},
                },
                "additionalProperties": False,
            },
        },
    },
    "additionalProperties": False,
}
# (en) STRICT prompt that spells out the schema — Session 3 contrasts it with a LOOSE prompt live.
# (kr) 스키마를 명시한 STRICT 프롬프트 — Session 3에서 LOOSE 프롬프트와 라이브 대비한다.
SYSTEM = (
    "Extract action items from the Korean meeting minutes. Output ONLY a JSON object with keys "
    "meeting_title (string) and action_items (array). Each action item MUST have exactly these keys: "
    "assignee (string), description (string), due (string), priority (one of high, medium, low). "
    "Use Korean for text values; infer a reasonable priority."
)
# (en) Three raw strings that mirror what a LOOSE prompt actually makes EXAONE emit (measured — see Session 3):
#      clean JSON / markdown-fenced JSON / key-renamed JSON.
# (kr) 느슨한 프롬프트일 때 EXAONE이 실제로 내는 출력 모양(측정값 — Session 3 참고)을 본뜬 원문 문자열 3종:
#      깔끔한 JSON / 마크다운 펜스 / 키 변형 JSON.
_good = {"meeting_title": "주간 백엔드 동기화", "action_items": [
    {"assignee": "이서연", "description": "idempotency 패치 머지", "due": "금요일", "priority": "high"}]}
raw_cases = [
    ("깔끔한 JSON", json.dumps(_good, ensure_ascii=False)),
    ("```json 펜스", "```json\n" + json.dumps(_good, ensure_ascii=False) + "\n```"),
    ("키 변형(meeting/task)", json.dumps(
        {"meeting": "주간 백엔드 동기화", "action_items": [
            {"assignee": "이서연", "task": "패치 머지", "due": "금요일", "priority": "high"}]},
        ensure_ascii=False)),
]
mech_results = []
for label, raw in raw_cases:
    # (en) Contrast naive json.loads (crashes on fences, accepts wrong keys) with the pipeline.
    # (kr) naive json.loads(펜스엔 깨지고 키 틀려도 통과)와 파이프라인을 대비한다.
    try:
        json.loads(raw)
        naive = "ok"
    except Exception as exc:
        naive = f"✗ {type(exc).__name__}"
    out = exaone.output.StructuredOutputPipeline(json_schema=ACTION_SCHEMA).process(raw)
    mech_results.append({"case": label, "json_loads": naive, "pipeline_ok": out.success,
                         "error": ((out.error or "").splitlines() or [""])[0][:45]})
print("json.loads vs 파이프라인:", mech_results)

**출력 해석:** `json.loads`는 두 방향으로 틀리고, 파이프라인은 둘 다 바로잡습니다 — 이게 "원문 파싱이 아니라 파이프라인"인 이유입니다.

- `깔끔한 JSON`: 둘 다 OK (이때는 차이 없음).
- ` ```json 펜스` → **`json.loads` ✗ JSONDecodeError**(백틱은 JSON 아님), **파이프라인 `ok=True`**: `JsonExtractor`가 펜스·산문에서 JSON을 **추출**합니다 — 모델이 흔히 내는 형식 잡음을 **재프롬프트 없이** 흡수합니다. 이게 파이프라인 고유의 일.
- `키 변형(meeting/task)` → **`json.loads` ok**(파싱은 됨), **파이프라인 `ok=False`** (`'meeting_title' is a required property`): 파싱만 하면 키가 틀린 잘못된 구조 를 그대로 받지만, 파이프라인은 스키마로 **검증·거부**하고 무엇이 틀렸는지 알려줍니다.
- 정리: 파이프라인 = **추출(형식 잡음 흡수) + 검증(위반 탐지)**. 펜스/산문 같은 형식 잡음은 파이프라인이 알아서 처리하고, 키/enum 같은 **의미 위반만** Session 3 에서 재프롬프트로 고칩니다 — 둘은 보완 관계입니다.

## Session 3. 라이브: 추출 → 검증 → 복구 (STRICT → LOOSE → REPAIR)


### Session 3-1. STRICT → LOOSE → REPAIR — 깨지면 복구하는 전체 루프

**하는 일:** 같은 회의록(m01)을 STRICT/LOOSE 프롬프트로 추출하고, LOOSE가 깨지면 **검증 에러를 모델에 다시 전달해 재요청(repair)** 해 복구합니다.

**정상(키 있을 때):**
- `STRICT → {'success': True, 'count_ok': True, 'head': '{...'}`
- `LOOSE  → {'success': False, 'fenced': True, 'error': "...required property", 'head': '```json ...'}`
- `REPAIR → {'success': True, 'fenced': False, 'head': '{"meeting_title": ...'}`

키가 없으면 `live: skip (no key)`가 출력됩니다.

**의미:** 깨짐 → 검증으로 **탐지** → 에러를 다시 전달해 **복구**의 운영 루프를 한눈에 봅니다.

In [ ]:
# (en) LOOSE prompt: asks for JSON but does NOT spell out the schema (measured to induce fences/key-drift).
# (kr) LOOSE 프롬프트: JSON은 요청하되 스키마를 명시하지 않는다(펜스·키 변형을 유발함이 측정됨).
LOOSE = "회의록에서 액션 아이템을 JSON으로 정리해줘."


def extract(system_prompt, user_text):
    # (en) Run one extraction; return the raw model text and the pipeline verdict.
    # (kr) 한 번 추출을 돌려 원문 모델 텍스트와 파이프라인 판정을 함께 반환한다.
    resp = client.chat(
        messages=[
            exaone.llm.ExaoneMessage(role="system", content=system_prompt),
            exaone.llm.ExaoneMessage(role="user", content=user_text),
        ],
        # (en) Rely on the EXAONE sampling defaults (temp=1.0/top_p=0.95/do_sample=True); only cap length.
        # (kr) EXAONE 샘플링 기본값(temp=1.0/top_p=0.95/do_sample=True)을 따르고 길이만 제한한다.
        options=exaone.llm.ExaoneGenerateOptions(max_new_tokens=512),
    )
    raw = resp.content or ""
    return raw, exaone.output.StructuredOutputPipeline(json_schema=ACTION_SCHEMA).process(raw)


def repair(error, minutes):
    # (en) Fix a rejected output by FEEDING THE VALIDATION ERROR + the exact schema back to the model.
    #      String repair can't fix key/enum errors (the library says so), so we re-prompt the model.
    # (kr) 거부된 출력을 복구한다: 검증 에러 + 정확한 스키마를 모델에 다시 전달해 재요청한다.
    #      문자열 repair로는 키/enum 오류를 복구하지 못하므로(라이브러리 명시) 재프롬프트한다.
    repair_system = (
        "Your previous JSON was rejected by schema validation. Fix it. Output ONLY a JSON object "
        "(no markdown fences, no prose) matching EXACTLY this JSON Schema:\n"
        + json.dumps(ACTION_SCHEMA, ensure_ascii=False)
    )
    repair_user = f"회의록:\n{minutes}\n\n이전 출력의 검증 오류: {error}\n스키마에 맞는 JSON만 다시 출력해줘."
    return extract(repair_system, repair_user)


live = None
if HAS_API and _samples:
    try:
        s = _samples[0]
        strict_raw, strict = extract(SYSTEM, s["minutes"])
        loose_raw, loose = extract(LOOSE, s["minutes"])
        n_strict = len((strict.data or {}).get("action_items", [])) if strict.success else 0
        live = {
            "strict": {"success": strict.success, "count_ok": n_strict == s.get("expected_action_count"),
                       "head": strict_raw[:60]},
            "loose": {"success": loose.success, "fenced": "```" in loose_raw,
                      "error": ((loose.error or "").splitlines() or [""])[0][:50], "head": loose_raw[:60]},
        }
        # (en) If LOOSE broke, repair it by feeding the validation error back (re-prompt, not string repair).
        # (kr) LOOSE가 깨졌으면 검증 에러를 다시 전달해 복구한다(문자열 repair 아님, 재프롬프트).
        if not loose.success:
            repair_raw, repaired = repair(loose.error or "", s["minutes"])
            live["repair"] = {"success": repaired.success, "fenced": "```" in repair_raw, "head": repair_raw[:60]}
    except Exception as exc:
        live = {"error": str(exc)[:200]}
if live is None:
    print("live: skip (no key)")
elif "error" in live:
    print("live error:", live["error"])
else:
    print("STRICT →", live["strict"])
    print("LOOSE  →", live["loose"])
    if "repair" in live:
        print("REPAIR →", live["repair"])

**출력 해석:** 같은 회의록인데 프롬프트가 출력 차이를 만들고, **깨지면 복구**합니다 — 추출 → 검증 → 복구의 전체 루프입니다.

- **STRICT** → `success=True`, `count_ok=True`: 스키마를 명시하면 모델이 `{` 로 시작하는 순응 JSON을 바로 냅니다(측정 3/3 깔끔).
- **LOOSE** → 보통 `success=False`, `fenced=True`: 느슨하면 ` ```json ` 펜스·키 변형(`meeting_title`→`meeting`)을 내어, 파이프라인이 펜스를 벗겨도 스키마에서 거부합니다 — `error`가 **무엇이 틀렸는지** 알려줍니다.
- **REPAIR** → `success=True`: 그 `error`와 정확한 스키마를 **모델에 다시 전달해 재요청**하면 모델이 순응 JSON으로 수정합니다(측정 3/3 복구). 파이프라인의 문자열 repair로는 키/enum을 처리할 수 없으므로(라이브러리가 그렇게 명시), 의미 위반의 복구는 **재프롬프트**가 정답입니다.
- **파이프라인이 한 일(중요):** LOOSE 출력의 **펜스를 벗겨 JSON을 꺼내고**(`json.loads`였다면 여기서 깨졌을 것) 위반을 **`error`로 진단**했기에 재프롬프트할 수 있었습니다 — 복구의 *실행*은 재프롬프트지만, 이를 *가능하게 한 것은* 파이프라인의 추출·검증입니다(Session 2의 `json.loads` 대비 참고).
- 정리: **프롬프트(1차) → 파이프라인이 추출·검증으로 탐지(2차) → 에러 다시 전달해 복구(3차)** 가 신뢰할 수 있는 구조화 추출의 운영 루프입니다. 비결정 샘플링이라 LOOSE가 드물게 바로 통과하면 복구 단계는 생략됩니다. 키가 없으면 `live: skip (no key)`.

## Session 4. 패키지


### Session 4-1. 패키지

**하는 일:** 골든 회귀(정적 M1/M6/M9 데모) 수치를 화면에 찍고, 파이프라인 메커니즘·라이브 STRICT/LOOSE 결과·SLO·trace 를 한 파일로 저장합니다.

**정상:** 두 줄이 출력됩니다 —
- `정적 메트릭 데모 (골든 25행 = 공통 'all' 22 + '02' 전용 3): M1=0.571 | M6_loose=0.500 | M9_stub=0.528`
- `saved …/_out/02/capstone_package.json` (해당 경로에 패키지 파일 생성)

**의미:** 캡스톤 제출·회귀용 결과를 한 파일로 묶고, 메트릭 수치를 패키지 안에만 묻지 않고 노출합니다.

In [ ]:
regression = regression_m1_m6_m9(load_capstone_golden("02"))
# (en) Surface the static metric numbers (else they would only live inside the package file).
# (kr) 정적 메트릭 수치를 화면에 노출한다(안 그러면 패키지 파일 안에만 남는다).
print(f"정적 메트릭 데모 (골든 {regression['n']}행 = 공통 'all' 22 + '02' 전용 3): "
      f"M1={regression['M1_mean']:.3f} | M6_loose={regression['M6_loose_mean']:.3f} | M9_stub={regression['M9_mean']:.3f}")
_live_ran = live is not None and "error" not in (live or {})
pkg_path = save_package("02", {
    "pipeline_mechanics": mech_results,
    "live_extract_repair": live,
    "regression": regression,
    "session_trace": [
        {"event": "pipeline_mechanics", "n": len(mech_results)},
        {"event": "live_extract_repair", "ran": _live_ran},
    ],
})

**출력 해석:** 메트릭 한 줄과 `saved …/_out/02/capstone_package.json`이 보이면 패키지가 완성된 것입니다.

- **회귀 행 구성(중요):** `load_capstone_golden("02")`는 공통 `all` 22행 + 이 캡스톤 전용 `02` 3행 = **25행**을 채점합니다. 02 전용 행(`mm01` 스키마 통과·`mm02` 필수 키 누락·`mm03` 충실도)이 회의록·액션 도메인을 대표하고, 나머지 22행은 메트릭 동작을 보여주는 공통 fixture입니다. (02 전용 행이 없으면 공통 22행만 조용히 채점되므로, 캡스톤 고유 행을 늘리는 것을 권장합니다.)
- **수치 의미:** `M1=0.571`(expected_answer 가 있는 7행만 엄격 일치 채점), `M6_loose=0.500`(required_keys 가 있는 8행의 느슨한 키 충족률 — `mm01` 통과·`mm02` 누락), `M9_stub=0.528`(grounding_context 가 있는 6행). 이건 **정적 fixture 메트릭 데모**라 메트릭 계산 동작일 뿐, 만든 에이전트의 실제 추출 성능이 아닙니다 — 특히 M9(`LengthRatioJudge`)는 테스트 전용 스텁이라 충실도가 아니라 길이비 근사입니다.
- **에이전트 신호는 따로:** 실제 동작은 패키지의 `pipeline_mechanics`(파이프라인 추출·검증·거부)·`live_extract_repair`(STRICT→LOOSE→복구 라이브 루프)에 담깁니다 — 회귀 데모 값과 섞어 보지 마세요.
- `save_package`가 절대경로(`TRACK10 / "_out" / "02"`)로 저장하므로 커널 CWD와 무관하게 항상 같은 위치에 저장됩니다.

## 마무리

이 캡스톤에서는 자유 서술 회의록을 검증 가능한 `action_items` JSON으로 추출하고, **STRICT → LOOSE → REPAIR** 흐름으로 복구까지 확인한 뒤 `_out/02/capstone_package.json`으로 저장했습니다.

**핵심 정리**
- **프롬프트:** 스키마를 명확히 쓰면 모델이 바로 순응 JSON을 낼 가능성이 높습니다.
- **검증:** `json.loads`는 펜스에는 깨지고 키 변형은 통과시킬 수 있습니다. `StructuredOutputPipeline`은 JSON을 추출하고 스키마 위반을 진단합니다.
- **복구:** 키·enum 같은 의미 위반은 문자열 보정보다 재프롬프트가 적합합니다. 검증 오류와 정확한 스키마를 다시 전달해 고칩니다.
- **패키지:** 파이프라인 동작 원리, 라이브 추출/복구 결과, 정적 메트릭을 한 파일로 묶습니다.

**한계**
- Session 2의 입력은 실제 모델 출력 모양을 본뜬 합성 예시입니다. 실제 모델 동작은 Session 3에서 확인합니다.
- LOOSE 출력은 비결정적입니다. 바로 통과하면 복구 단계가 생략될 수 있으며, 운영에서는 재시도 횟수 상한이 필요합니다.
- M9(`LengthRatioJudge`)는 테스트 전용 스텁입니다. 정적 22행은 메트릭 동작 fixture이고, 도메인 신호는 `02` 전용 3행과 라이브 경로에서 봅니다.

**다음:** `10_03` Code Review Assistant. 운영화는 `10_07` 프로덕션 하네스를 참고하세요.

## 체크포인트

- [ ] Session 2 `json.loads` vs 파이프라인 비교 확인
- [ ] Session 3 STRICT 통과 · LOOSE 실패 · REPAIR 복구 확인(키 있을 때)
- [ ] Session 4 메트릭 수치 출력 + `_out/02/capstone_package.json` 저장
